
# Covariate screening for 41 pair–distribution models

This notebook refits each of the 41 retained pairwise distributions with **one
covariate at a time** using a log-scale accelerated failure-time (AFT) model:

\[
T_i = \exp(\beta z_i)Y_i,
\qquad
Y_i \sim F_{\theta}.
\]

For each pair–distribution–covariate combination it reports:

- baseline AIC and covariate-model AIC;
- \(\Delta\mathrm{AIC}=\mathrm{AIC}_{0}-\mathrm{AIC}_{1}\), so a positive
  value favours covariate incorporation;
- likelihood-ratio test statistic and p-value for \(H_0:\beta=0\);
- the AFT coefficient and acceleration ratio \(\exp(\beta)\);
- conditional probability-integral-transform KS and Anderson–Darling
  statistics;
- parametric-bootstrap KS and AD p-values, **refitting the same model in every
  bootstrap replicate**;
- whether an AIC improvement of at least 2 is achieved without turning a
  previously accepted KS or AD result into a rejection.

The main exported table has one row per pair–distribution model and one
\(\Delta\)AIC column per covariate, matching the requested structure.

**Runtime note.** The default 499 bootstrap replicates require many model
refits. Use 999 or 1,999 replicates for final manuscript estimates if runtime
permits. For a quick code check, temporarily set `N_BOOT = 19`.


In [2]:

# ============================================================
# 1. Imports and analysis configuration
# ============================================================

from pathlib import Path
import hashlib
import io
import os
import time
import warnings

import numpy as np
import pandas as pd
from scipy import optimize, stats

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ---------- file paths ----------
# The first existing path is used. You normally only need data3.xlsx in the
# same folder as this notebook.
DATA_CANDIDATES = [
    Path(os.environ.get("HEADWAY_DATA_PATH", "data3.xlsx")),
    Path(r"D:\Headway\data3.xlsx"),
    Path("project_sources/12-data3.xlsx"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])

OUTPUT_DIR = Path(
    os.environ.get(
        "HEADWAY_OUTPUT_DIR",
        str(DATA_PATH.parent / "Tables"),
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_XLSX = OUTPUT_DIR / "T4_41_Distribution_Covariate_Screening.xlsx"


# ---------- statistical settings ----------
ALPHA_GOF = 0.05
DELTA_AIC_REQUIRED = 2.0

# 499 -> minimum attainable add-one-corrected p-value = 1/500 = 0.002.
# Increase to 999 or 1999 for the final manuscript if runtime permits.
N_BOOT = int(os.environ.get("HEADWAY_N_BOOT", "499"))

RANDOM_SEED = 20260730
MIN_VALID_BOOT_FRACTION = 0.80
MIN_BINARY_GROUP_N = 5

# Site coding is explicit so the coefficient sign is reproducible.
# Site = 1 for Tikatuli and 0 for Shahjahanpur.
SITE_ONE_LEVEL = "Tikatuli"

# Optional testing hook. Leave at 0 to run all 41 models.
MODEL_LIMIT = int(os.environ.get("HEADWAY_MODEL_LIMIT", "0"))


# ---------- variable definitions ----------
HEADWAY = "Time_Headway"

COVARIATES = [
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Occupancy",
    "Off_centeredness",
    "Site",
    "Flow_pcu/hr/m",
]

COVARIATE_LABELS = {
    "Target_Speed_km/hr": "Target Vehicle Speed",
    "Leading_Speed_km/hr": "Leading Vehicle Speed",
    "Speed_Difference": "Speed Difference",
    "Occupancy": "Occupancy",
    "Off_centeredness": "Off-centeredness",
    "Site": "Site",
    "Flow_pcu/hr/m": "Flow",
}

CONTINUOUS_COVARIATES = {
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Flow_pcu/hr/m",
}

BINARY_COVARIATES = {
    "Occupancy",
    "Off_centeredness",
    "Site",
}

print("Data path :", DATA_PATH)
print("Output    :", OUTPUT_XLSX)
print("Bootstrap :", N_BOOT, "replicates per fitted model")


Data path : D:\Headway\data3.xlsx
Output    : D:\Headway\Tables\T4_41_Distribution_Covariate_Screening.xlsx
Bootstrap : 499 replicates per fitted model



## Distribution list supplied for screening

`Reference AIC` is retained as an audit field. The notebook independently
refits every baseline model and checks that its AIC agrees with the supplied
value.


In [3]:

# ============================================================
# 2. Exact list of 41 pair–distribution models
# ============================================================

MODEL_TSV = """Pair\tDistribution\tReference AIC
PR_following_4W\tWeibull\t130.4217082
PR_following_4W\tGeneralized gamma\t130.432732
PR_following_4W\tPearson type III\t133.2201103
PR_following_4W\tGamma\t134.7090156
PR_following_4W\tLog-normal\t138.6047134
PR_following_4W\tInverse Gaussian\t138.8883636
PR_following_MT_3W\tWeibull\t278.6461759
PR_following_MT_3W\tGeneralized gamma\t280.0712218
PR_following_MT_3W\tPearson type III\t282.3900634
PR_following_MT_3W\tGamma\t287.727773
PR_following_NMT_3W\tWeibull\t140.8386151
PR_following_NMT_3W\tGeneralized gamma\t141.3538015
PR_following_NMT_3W\tPearson type III\t143.6392699
PR_following_NMT_3W\tGamma\t146.0173829
BTW_following_4W\tGeneralized gamma\t623.6222796
BTW_following_4W\tWeibull\t624.5270333
BTW_following_4W\tPearson type III\t625.1180291
BTW_following_4W\tGamma\t625.5623798
BTW_following_MT_3W\tPearson type III\t442.2116007
BTW_following_MT_3W\tGamma\t442.4850603
BTW_following_MT_3W\tLog-normal\t443.3417007
BTW_following_MT_3W\tInverse Gaussian\t443.5417504
BTW_following_MT_3W\tGeneralized gamma\t443.5420673
BTW_following_MT_2W\tPearson type III\t167.4725515
BTW_following_MT_2W\tInverse Gaussian\t168.1679223
BTW_following_MT_2W\tGamma\t168.2655264
BTW_following_MT_2W\tLog-normal\t168.8096561
BTW_following_MT_2W\tGeneralized gamma\t170.0952509
BTW_following_MT_2W\tWeibull\t171.8041123
BTW_following_NMT_3W\tPearson type III\t296.8282305
BTW_following_NMT_3W\tGamma\t298.009388
BTW_following_NMT_3W\tInverse Gaussian\t298.0462949
BTW_following_NMT_3W\tLog-normal\t299.2474746
BTW_following_NMT_3W\tGeneralized gamma\t299.8704794
BTW_following_NMT_3W\tWeibull\t301.901369
BTW_following_NMT_2W\tInverse Gaussian\t94.16949926
BTW_following_NMT_2W\tLog-normal\t94.47462234
BTW_following_NMT_2W\tGamma\t94.56410458
BTW_following_NMT_2W\tPearson type III\t95.16679216
BTW_following_NMT_2W\tGeneralized gamma\t96.29536501
BTW_following_NMT_2W\tWeibull\t96.87733508"""

BASELINE_MODELS = pd.read_csv(io.StringIO(MODEL_TSV), sep="\t")
BASELINE_MODELS.insert(0, "Model order", np.arange(1, len(BASELINE_MODELS) + 1))

if len(BASELINE_MODELS) != 41:
    raise ValueError(f"Expected 41 models, found {len(BASELINE_MODELS)}")

if BASELINE_MODELS.duplicated(["Pair", "Distribution"]).any():
    raise ValueError("The model list contains duplicate pair–distribution rows.")

MODELS_TO_RUN = (
    BASELINE_MODELS.head(MODEL_LIMIT).copy()
    if MODEL_LIMIT > 0
    else BASELINE_MODELS.copy()
)

print(f"Models configured: {len(BASELINE_MODELS)}")
print(f"Models to run     : {len(MODELS_TO_RUN)}")
display(BASELINE_MODELS)


Models configured: 41
Models to run     : 41


,Model order,Pair,Distribution,Reference AIC
0,1,PR_following_4W,Weibull,130.421708
1,2,PR_following_4W,Generalized gamma,130.432732
2,3,PR_following_4W,Pearson type III,133.220110
3,4,PR_following_4W,Gamma,134.709016
4,5,PR_following_4W,Log-normal,138.604713
5,6,PR_following_4W,Inverse Gaussian,138.888364
6,7,PR_following_MT_3W,Weibull,278.646176
7,8,PR_following_MT_3W,Generalized gamma,280.071222
8,9,PR_following_MT_3W,Pearson type III,282.390063
9,10,PR_following_MT_3W,Gamma,287.727773


In [4]:

# ============================================================
# 3. Load and validate data
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "data3.xlsx was not found. Put it beside this notebook or edit DATA_CANDIDATES."
    )

raw_df = pd.read_excel(DATA_PATH, sheet_name=0)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

REQUIRED_COLUMNS = ["Pair", HEADWAY] + COVARIATES
missing_columns = [c for c in REQUIRED_COLUMNS if c not in raw_df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

df = raw_df.copy()

# Numeric conversion is explicit. The original raw workbook is not modified.
for column in [HEADWAY] + sorted(CONTINUOUS_COVARIATES):
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Use one common complete-case dataset so every covariate comparison within a
# pair has the same n and the AIC values are directly comparable.
valid = df[REQUIRED_COLUMNS].notna().all(axis=1)
valid &= np.isfinite(df[HEADWAY])
valid &= df[HEADWAY] > 0
for column in CONTINUOUS_COVARIATES:
    valid &= np.isfinite(df[column])

analysis_df = df.loc[valid].copy()

listed_pairs = set(BASELINE_MODELS["Pair"])
available_pairs = set(analysis_df["Pair"].astype(str))
missing_pairs = sorted(listed_pairs - available_pairs)
if missing_pairs:
    raise ValueError(f"These listed pairs are absent from data: {missing_pairs}")

unexpected_sites = sorted(
    set(analysis_df["Site"].astype(str).unique())
    - {"Tikatuli", "Shahjahanpur"}
)
if unexpected_sites:
    raise ValueError(
        "Unexpected Site values found: "
        f"{unexpected_sites}. Update the explicit site coding before analysis."
    )

pair_counts = (
    analysis_df.loc[analysis_df["Pair"].isin(listed_pairs)]
    .groupby("Pair", observed=True)
    .size()
    .rename("n")
    .reset_index()
)

print(f"Raw rows              : {len(raw_df)}")
print(f"Complete analysis rows: {len(analysis_df)}")
print(f"Rows excluded         : {len(raw_df) - len(analysis_df)}")
display(pair_counts)


Raw rows              : 898
Complete analysis rows: 898
Rows excluded         : 0


,Pair,n
0,BTW_following_4W,250
1,BTW_following_MT_2W,73
2,BTW_following_MT_3W,186
3,BTW_following_NMT_2W,41
4,BTW_following_NMT_3W,119
5,PR_following_4W,43
6,PR_following_MT_3W,104
7,PR_following_NMT_3W,49



## Model parameterization and fitting

The five positive-support families retain `loc = 0`, exactly as in the supplied
baseline notebook. Pearson type III retains a freely estimated location.

Continuous covariates are standardized **within each pair**, so
\(\exp(\beta)\) is the multiplicative headway ratio for a one-standard-deviation
increase. Boolean covariates use 0/1 coding. `Site` uses
Shahjahanpur = 0 and Tikatuli = 1.


In [5]:

# ============================================================
# 4. Distribution definitions and AFT maximum likelihood
# ============================================================

DIST_SPECS = {
    "gengamma": {
        "label": "Generalized gamma",
        "distribution": stats.gengamma,
        "mode": "floc0",
    },
    "weibull_min": {
        "label": "Weibull",
        "distribution": stats.weibull_min,
        "mode": "floc0",
    },
    "pearson3": {
        "label": "Pearson type III",
        "distribution": stats.pearson3,
        "mode": "free",
    },
    "gamma": {
        "label": "Gamma",
        "distribution": stats.gamma,
        "mode": "floc0",
    },
    "lognorm": {
        "label": "Log-normal",
        "distribution": stats.lognorm,
        "mode": "floc0",
    },
    "invgauss": {
        "label": "Inverse Gaussian",
        "distribution": stats.invgauss,
        "mode": "floc0",
    },
}

LABEL_TO_KEY = {
    specification["label"]: key
    for key, specification in DIST_SPECS.items()
}


def stable_seed(*parts):
    """Stable seed independent of Python's session-randomized hash()."""
    token = "|".join(map(str, parts)).encode("utf-8")
    offset = int(hashlib.sha256(token).hexdigest()[:8], 16)
    return int((RANDOM_SEED + offset) % (2**32 - 1))


def fit_baseline(key, x):
    """Baseline MLE using the exact conventions that generated the 41 AICs."""
    x = np.asarray(x, dtype=float)
    specification = DIST_SPECS[key]
    distribution = specification["distribution"]

    if specification["mode"] == "free":
        parameters = tuple(np.asarray(distribution.fit(x), dtype=float))
        k = len(parameters)
    else:
        parameters = tuple(
            np.asarray(distribution.fit(x, floc=0.0), dtype=float)
        )
        k = len(parameters) - 1

    log_likelihood = float(np.sum(distribution.logpdf(x, *parameters)))
    if not np.isfinite(log_likelihood):
        raise FloatingPointError(f"Non-finite baseline likelihood for {key}")

    return {
        "parameters": parameters,
        "beta": 0.0,
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": True,
        "optimizer": "scipy.fit",
    }


def theta_from_parameters(key, parameters, beta=0.0):
    """
    Convert SciPy parameters to a numerically stable optimizer vector.

    Positive shape/scale parameters are optimized on the log scale.
    The generalized-gamma exponent is restricted to c > 0, matching the
    positive-c solution used for all supplied baseline AIC values.
    """
    p = tuple(float(v) for v in parameters)

    if key == "gengamma":
        a, c, _, scale = p
        return np.array([np.log(a), np.log(abs(c)), np.log(scale), beta], float)
    if key in {"weibull_min", "gamma", "lognorm", "invgauss"}:
        shape, _, scale = p
        return np.array([np.log(shape), np.log(scale), beta], float)
    if key == "pearson3":
        skew, loc, scale = p
        return np.array([skew, loc, np.log(scale), beta], float)

    raise KeyError(key)


def parameters_from_theta(key, theta):
    """Return (SciPy parameter tuple, beta) from an optimizer vector."""
    theta = np.asarray(theta, dtype=float)

    if key == "gengamma":
        parameters = (
            float(np.exp(theta[0])),
            float(np.exp(theta[1])),
            0.0,
            float(np.exp(theta[2])),
        )
        beta = float(theta[3])
    elif key in {"weibull_min", "gamma", "lognorm", "invgauss"}:
        parameters = (
            float(np.exp(theta[0])),
            0.0,
            float(np.exp(theta[1])),
        )
        beta = float(theta[2])
    elif key == "pearson3":
        parameters = (
            float(theta[0]),
            float(theta[1]),
            float(np.exp(theta[2])),
        )
        beta = float(theta[3])
    else:
        raise KeyError(key)

    return parameters, beta


def optimizer_bounds(key):
    """Broad but finite bounds that prevent numerical overflow."""
    log_positive = (-12.0, 12.0)
    beta_bound = (-3.0, 3.0)

    if key == "gengamma":
        return [log_positive, log_positive, (-20.0, 20.0), beta_bound]
    if key in {"weibull_min", "gamma", "lognorm", "invgauss"}:
        return [log_positive, (-20.0, 20.0), beta_bound]
    if key == "pearson3":
        return [(-50.0, 50.0), (None, None), (-20.0, 20.0), beta_bound]
    raise KeyError(key)


def aft_negative_loglik(theta, key, x, z):
    """
    Negative conditional log-likelihood under
        T_i = exp(beta*z_i) * Y_i.
    """
    try:
        parameters, beta = parameters_from_theta(key, theta)
        eta = np.clip(beta * z, -50.0, 50.0)
        adjusted = x * np.exp(-eta)
        distribution = DIST_SPECS[key]["distribution"]
        log_density = (
            distribution.logpdf(adjusted, *parameters)
            - eta
        )
    except Exception:
        return 1e100

    if not np.all(np.isfinite(log_density)):
        return 1e100

    return float(-np.sum(log_density))


def fit_aft(key, x, z, start_theta=None, robust=True):
    """
    Fit one-covariate AFT model.

    The null-model solution beta=0 is always retained as a candidate. Therefore,
    numerical noise cannot produce a covariate-model likelihood below the
    corresponding baseline likelihood.
    """
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    baseline = fit_baseline(key, x)
    null_theta = theta_from_parameters(
        key,
        baseline["parameters"],
        beta=0.0,
    )

    starts = [null_theta]

    if start_theta is not None:
        starts.insert(0, np.asarray(start_theta, dtype=float))

    if robust and np.std(z) > 0:
        slope = float(
            np.cov(np.log(x), z, ddof=1)[0, 1]
            / np.var(z, ddof=1)
        )
        slope = float(np.clip(slope, -1.5, 1.5))
        for value in [slope, -slope, 0.25, -0.25]:
            candidate = null_theta.copy()
            candidate[-1] = value
            starts.append(candidate)

    candidates = [
        {
            "theta": null_theta,
            "fun": aft_negative_loglik(null_theta, key, x, z),
            "success": True,
            "message": "Nested null candidate",
            "method": "null",
        }
    ]

    bounds = optimizer_bounds(key)

    for initial in starts:
        try:
            result = optimize.minimize(
                aft_negative_loglik,
                initial,
                args=(key, x, z),
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "maxiter": 4000 if robust else 1500,
                    "ftol": 1e-10 if robust else 1e-8,
                    "gtol": 1e-7 if robust else 1e-5,
                    "maxls": 50,
                },
            )
        except Exception:
            continue

        if np.isfinite(result.fun):
            candidates.append(
                {
                    "theta": np.asarray(result.x, dtype=float),
                    "fun": float(result.fun),
                    "success": bool(result.success),
                    "message": str(result.message),
                    "method": "L-BFGS-B",
                }
            )

    best = min(candidates, key=lambda item: item["fun"])
    parameters, beta = parameters_from_theta(key, best["theta"])
    log_likelihood = float(-best["fun"])
    k = baseline["k"] + 1

    return {
        "parameters": parameters,
        "beta": beta,
        "theta": np.asarray(best["theta"], dtype=float),
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": bool(best["success"]),
        "optimizer": best["method"],
        "optimizer_message": best["message"],
    }


def encode_covariate(pair_data, covariate):
    """
    Return z, coding note, and estimability information.

    Continuous variables are z-scored within the current pair.
    """
    series = pair_data[covariate]

    if covariate in CONTINUOUS_COVARIATES:
        values = pd.to_numeric(series, errors="coerce").to_numpy(float)
        mean = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        if not np.isfinite(sd) or sd <= 1e-12:
            return values * np.nan, "Constant within pair", False
        z = (values - mean) / sd
        note = f"Within-pair z-score; mean={mean:.6g}, SD={sd:.6g}"
        return z, note, True

    if covariate in {"Occupancy", "Off_centeredness"}:
        if pd.api.types.is_bool_dtype(series):
            z = series.astype(float).to_numpy()
        else:
            normalized = series.astype(str).str.strip().str.lower()
            mapping = {
                "true": 1.0, "false": 0.0,
                "yes": 1.0, "no": 0.0,
                "1": 1.0, "0": 0.0,
            }
            z = normalized.map(mapping).to_numpy(float)

        if not np.all(np.isfinite(z)):
            return z, "Unrecognized binary coding", False

        counts = pd.Series(z).value_counts()
        estimable = (
            set(counts.index) == {0.0, 1.0}
            and int(counts.min()) >= MIN_BINARY_GROUP_N
        )
        note = (
            f"False=0, True=1; n0={(z == 0).sum()}, n1={(z == 1).sum()}"
        )
        return z, note, bool(estimable)

    if covariate == "Site":
        values = series.astype(str).to_numpy()
        z = (values == SITE_ONE_LEVEL).astype(float)
        counts = pd.Series(z).value_counts()
        estimable = (
            set(counts.index) == {0.0, 1.0}
            and int(counts.min()) >= MIN_BINARY_GROUP_N
        )
        note = (
            f"Shahjahanpur=0, {SITE_ONE_LEVEL}=1; "
            f"n0={(z == 0).sum()}, n1={(z == 1).sum()}"
        )
        return z, note, bool(estimable)

    raise KeyError(covariate)



## Conditional goodness-of-fit tests

For observation \(i\), the fitted conditional CDF is

\[
u_i =
F_{\hat\theta}\!\left(T_i\exp(-\hat\beta z_i)\right).
\]

Under the fitted model, these PIT values should follow Uniform(0,1).
The bootstrap holds the observed covariate values fixed, simulates conditional
headways, refits the complete baseline or covariate model, and then recomputes
KS and AD. This accounts for parameter estimation.


In [6]:

# ============================================================
# 5. PIT-based KS/AD statistics and parametric bootstrap
# ============================================================

def uniform_ks_ad(u):
    """KS D and Anderson–Darling A² for values tested against Uniform(0,1)."""
    u = np.sort(np.clip(np.asarray(u, dtype=float), 1e-12, 1 - 1e-12))
    n = len(u)
    i = np.arange(1, n + 1)

    ks_d = max(
        np.max(i / n - u),
        np.max(u - (i - 1) / n),
    )

    ad_a2 = (
        -n
        - np.sum(
            (2 * i - 1)
            * (np.log(u) + np.log(1 - u[::-1]))
        )
        / n
    )

    return float(ks_d), float(ad_a2)


def conditional_gof_statistics(key, x, z, parameters, beta):
    """Observed conditional PIT statistics for one fitted model."""
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    eta = np.clip(beta * z, -50.0, 50.0)
    adjusted = x * np.exp(-eta)
    distribution = DIST_SPECS[key]["distribution"]
    u = distribution.cdf(adjusted, *parameters)
    if not np.all(np.isfinite(u)):
        raise FloatingPointError("Non-finite conditional CDF values")
    return uniform_ks_ad(u)


def bootstrap_gof(
    key,
    x,
    z,
    fitted_model,
    include_covariate,
    n_boot,
    seed,
):
    """
    Lilliefors-type parametric bootstrap for conditional KS and AD.

    Covariate values are held fixed. The same model is refitted in every
    replicate. Add-one p-value correction prevents an estimate of exactly zero.
    """
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    n = len(x)

    parameters = fitted_model["parameters"]
    beta = float(fitted_model["beta"])
    distribution = DIST_SPECS[key]["distribution"]

    observed_ks, observed_ad = conditional_gof_statistics(
        key,
        x,
        z,
        parameters,
        beta,
    )

    if n_boot <= 0:
        return {
            "KS D": observed_ks,
            "KS p": np.nan,
            "AD A²": observed_ad,
            "AD p": np.nan,
            "Bootstrap valid": 0,
            "Bootstrap requested": 0,
            "Bootstrap status": "Not requested",
        }

    rng = np.random.default_rng(seed)
    ks_exceedances = 0
    ad_exceedances = 0
    valid = 0

    eta = np.clip(beta * z, -50.0, 50.0)

    for _ in range(n_boot):
        try:
            baseline_draw = np.asarray(
                distribution.rvs(
                    *parameters,
                    size=n,
                    random_state=rng,
                ),
                dtype=float,
            )

            if (
                baseline_draw.shape != (n,)
                or not np.all(np.isfinite(baseline_draw))
            ):
                continue

            simulated_x = baseline_draw * np.exp(eta)

            if include_covariate:
                refit = fit_aft(
                    key,
                    simulated_x,
                    z,
                    start_theta=fitted_model["theta"],
                    robust=False,
                )
            else:
                refit = fit_baseline(key, simulated_x)

            bootstrap_ks, bootstrap_ad = conditional_gof_statistics(
                key,
                simulated_x,
                z,
                refit["parameters"],
                refit["beta"],
            )

            if not (
                np.isfinite(bootstrap_ks)
                and np.isfinite(bootstrap_ad)
            ):
                continue

        except Exception:
            continue

        valid += 1
        ks_exceedances += int(bootstrap_ks >= observed_ks)
        ad_exceedances += int(bootstrap_ad >= observed_ad)

    minimum_valid = int(np.ceil(MIN_VALID_BOOT_FRACTION * n_boot))

    if valid < minimum_valid:
        ks_p = np.nan
        ad_p = np.nan
        status = f"Insufficient valid replicates: {valid}/{n_boot}"
    else:
        ks_p = float((1 + ks_exceedances) / (1 + valid))
        ad_p = float((1 + ad_exceedances) / (1 + valid))
        status = "OK"

    return {
        "KS D": observed_ks,
        "KS p": ks_p,
        "AD A²": observed_ad,
        "AD p": ad_p,
        "Bootstrap valid": int(valid),
        "Bootstrap requested": int(n_boot),
        "Bootstrap status": status,
    }



## Run all 41 × 7 one-covariate refits

The baseline bootstrap is cached once per pair–distribution model. Every
covariate model receives its own conditional parametric bootstrap.


In [6]:

# ============================================================
# 6. Baseline audit and complete covariate screen
# ============================================================

analysis_start = time.time()
baseline_rows = []
screen_rows = []

baseline_cache = {}

for model_number, (_, model_row) in enumerate(
    MODELS_TO_RUN.iterrows(),
    start=1,
):
    pair = model_row["Pair"]
    distribution_label = model_row["Distribution"]
    reference_aic = float(model_row["Reference AIC"])
    model_order = int(model_row["Model order"])
    key = LABEL_TO_KEY[distribution_label]

    pair_data = analysis_df.loc[analysis_df["Pair"] == pair].copy()
    x = pair_data[HEADWAY].to_numpy(float)
    n = len(x)

    print(
        f"[Model {model_number:02d}/{len(MODELS_TO_RUN)}] "
        f"{pair} | {distribution_label} | n={n}"
    )

    baseline_fit = fit_baseline(key, x)
    baseline_gof = bootstrap_gof(
        key=key,
        x=x,
        z=np.zeros(n, dtype=float),
        fitted_model=baseline_fit,
        include_covariate=False,
        n_boot=N_BOOT,
        seed=stable_seed("baseline", pair, key),
    )

    baseline_fit = {
        **baseline_fit,
        **baseline_gof,
    }
    baseline_cache[(pair, key)] = baseline_fit

    aic_difference = baseline_fit["AIC"] - reference_aic
    baseline_rows.append(
        {
            "Model order": model_order,
            "Pair": pair,
            "Distribution key": key,
            "Distribution": distribution_label,
            "n": n,
            "k": baseline_fit["k"],
            "Reference AIC": reference_aic,
            "Refitted baseline AIC": baseline_fit["AIC"],
            "AIC audit difference": aic_difference,
            "AIC reconciled (|difference| ≤ 0.001)": (
                abs(aic_difference) <= 0.001
            ),
            "Baseline logLik": baseline_fit["logLik"],
            "Baseline parameters": ", ".join(
                f"{value:.8g}"
                for value in baseline_fit["parameters"]
            ),
            "Baseline KS D": baseline_fit["KS D"],
            "Baseline KS p": baseline_fit["KS p"],
            "Baseline AD A²": baseline_fit["AD A²"],
            "Baseline AD p": baseline_fit["AD p"],
            "Baseline KS accepted": (
                np.isfinite(baseline_fit["KS p"])
                and baseline_fit["KS p"] >= ALPHA_GOF
            ),
            "Baseline AD accepted": (
                np.isfinite(baseline_fit["AD p"])
                and baseline_fit["AD p"] >= ALPHA_GOF
            ),
            "Baseline accepted by both": (
                np.isfinite(baseline_fit["KS p"])
                and np.isfinite(baseline_fit["AD p"])
                and baseline_fit["KS p"] >= ALPHA_GOF
                and baseline_fit["AD p"] >= ALPHA_GOF
            ),
            "Baseline bootstrap valid": baseline_fit["Bootstrap valid"],
            "Baseline bootstrap requested": baseline_fit["Bootstrap requested"],
            "Baseline bootstrap status": baseline_fit["Bootstrap status"],
        }
    )

    for covariate_number, covariate in enumerate(COVARIATES, start=1):
        covariate_label = COVARIATE_LABELS[covariate]
        z, coding_note, estimable = encode_covariate(pair_data, covariate)

        print(
            f"    [{covariate_number}/{len(COVARIATES)}] "
            f"{covariate_label}",
            end="",
        )

        common = {
            "Model order": model_order,
            "Pair": pair,
            "Distribution key": key,
            "Distribution": distribution_label,
            "Covariate": covariate,
            "Covariate label": covariate_label,
            "n": n,
            "Coding": coding_note,
            "Estimable": bool(estimable),
            "Reference AIC": reference_aic,
            "Baseline k": baseline_fit["k"],
            "Baseline logLik": baseline_fit["logLik"],
            "Baseline AIC": baseline_fit["AIC"],
            "Baseline KS D": baseline_fit["KS D"],
            "Baseline KS p": baseline_fit["KS p"],
            "Baseline AD A²": baseline_fit["AD A²"],
            "Baseline AD p": baseline_fit["AD p"],
            "Baseline KS accepted": (
                np.isfinite(baseline_fit["KS p"])
                and baseline_fit["KS p"] >= ALPHA_GOF
            ),
            "Baseline AD accepted": (
                np.isfinite(baseline_fit["AD p"])
                and baseline_fit["AD p"] >= ALPHA_GOF
            ),
            "Baseline accepted by both": (
                np.isfinite(baseline_fit["KS p"])
                and np.isfinite(baseline_fit["AD p"])
                and baseline_fit["KS p"] >= ALPHA_GOF
                and baseline_fit["AD p"] >= ALPHA_GOF
            ),
        }

        if not estimable:
            screen_rows.append(
                {
                    **common,
                    "Covariate k": np.nan,
                    "Covariate logLik": np.nan,
                    "Covariate AIC": np.nan,
                    "ΔAIC": np.nan,
                    "β": np.nan,
                    "Acceleration ratio exp(β)": np.nan,
                    "LR χ²": np.nan,
                    "LR p": np.nan,
                    "Covariate KS D": np.nan,
                    "Covariate KS p": np.nan,
                    "ΔKS p": np.nan,
                    "Covariate AD A²": np.nan,
                    "Covariate AD p": np.nan,
                    "ΔAD p": np.nan,
                    "Covariate KS accepted": False,
                    "Covariate AD accepted": False,
                    "Covariate accepted by both": False,
                    "KS acceptance preserved": False,
                    "AD acceptance preserved": False,
                    "GOF acceptance preserved": False,
                    "AIC improvement ≥ 2": False,
                    "Passes stated rule": False,
                    "Strict pass (AIC + both GOF)": False,
                    "Decision": "Not estimable",
                    "Optimizer converged": False,
                    "Optimizer": "",
                    "Optimizer message": "",
                    "Bootstrap valid": 0,
                    "Bootstrap requested": N_BOOT,
                    "Bootstrap status": "Not run: covariate not estimable",
                }
            )
            print(" | not estimable")
            continue

        fit_start = time.time()

        covariate_fit = fit_aft(
            key=key,
            x=x,
            z=z,
            robust=True,
        )

        covariate_gof = bootstrap_gof(
            key=key,
            x=x,
            z=z,
            fitted_model=covariate_fit,
            include_covariate=True,
            n_boot=N_BOOT,
            seed=stable_seed("covariate", pair, key, covariate),
        )

        delta_aic = baseline_fit["AIC"] - covariate_fit["AIC"]
        lr_chi2 = max(
            2 * (covariate_fit["logLik"] - baseline_fit["logLik"]),
            0.0,
        )
        lr_p = float(stats.chi2.sf(lr_chi2, df=1))

        baseline_ks_accepted = common["Baseline KS accepted"]
        baseline_ad_accepted = common["Baseline AD accepted"]

        covariate_ks_accepted = (
            np.isfinite(covariate_gof["KS p"])
            and covariate_gof["KS p"] >= ALPHA_GOF
        )
        covariate_ad_accepted = (
            np.isfinite(covariate_gof["AD p"])
            and covariate_gof["AD p"] >= ALPHA_GOF
        )

        p_values_available = (
            np.isfinite(baseline_fit["KS p"])
            and np.isfinite(baseline_fit["AD p"])
            and np.isfinite(covariate_gof["KS p"])
            and np.isfinite(covariate_gof["AD p"])
        )

        ks_preserved = (
            p_values_available
            and ((not baseline_ks_accepted) or covariate_ks_accepted)
        )
        ad_preserved = (
            p_values_available
            and ((not baseline_ad_accepted) or covariate_ad_accepted)
        )
        gof_preserved = bool(ks_preserved and ad_preserved)

        aic_improved = bool(delta_aic >= DELTA_AIC_REQUIRED)
        covariate_both_accepted = bool(
            covariate_ks_accepted and covariate_ad_accepted
        )
        passes_rule = bool(aic_improved and gof_preserved)
        strict_pass = bool(aic_improved and covariate_both_accepted)

        if passes_rule:
            decision = "Pass"
        elif not aic_improved:
            decision = "Fail: ΔAIC < 2"
        elif not p_values_available:
            decision = "Fail: bootstrap p-value unavailable"
        else:
            decision = "Fail: accepted GOF result became rejected"

        screen_rows.append(
            {
                **common,
                "Covariate k": covariate_fit["k"],
                "Covariate logLik": covariate_fit["logLik"],
                "Covariate AIC": covariate_fit["AIC"],
                "ΔAIC": delta_aic,
                "β": covariate_fit["beta"],
                "Acceleration ratio exp(β)": float(
                    np.exp(covariate_fit["beta"])
                ),
                "LR χ²": lr_chi2,
                "LR p": lr_p,
                "Covariate KS D": covariate_gof["KS D"],
                "Covariate KS p": covariate_gof["KS p"],
                "ΔKS p": covariate_gof["KS p"] - baseline_fit["KS p"],
                "Covariate AD A²": covariate_gof["AD A²"],
                "Covariate AD p": covariate_gof["AD p"],
                "ΔAD p": covariate_gof["AD p"] - baseline_fit["AD p"],
                "Covariate KS accepted": covariate_ks_accepted,
                "Covariate AD accepted": covariate_ad_accepted,
                "Covariate accepted by both": covariate_both_accepted,
                "KS acceptance preserved": ks_preserved,
                "AD acceptance preserved": ad_preserved,
                "GOF acceptance preserved": gof_preserved,
                "AIC improvement ≥ 2": aic_improved,
                "Passes stated rule": passes_rule,
                "Strict pass (AIC + both GOF)": strict_pass,
                "Decision": decision,
                "Optimizer converged": covariate_fit["converged"],
                "Optimizer": covariate_fit["optimizer"],
                "Optimizer message": covariate_fit["optimizer_message"],
                "Bootstrap valid": covariate_gof["Bootstrap valid"],
                "Bootstrap requested": covariate_gof["Bootstrap requested"],
                "Bootstrap status": covariate_gof["Bootstrap status"],
            }
        )

        print(
            f" | ΔAIC={delta_aic:7.3f}"
            f" | KS p={covariate_gof['KS p']:.3f}"
            f" | AD p={covariate_gof['AD p']:.3f}"
            f" | {decision}"
            f" | {time.time() - fit_start:.1f}s"
        )


BASELINE_AUDIT = pd.DataFrame(baseline_rows).sort_values(
    "Model order",
    ignore_index=True,
)

FULL_SCREEN = pd.DataFrame(screen_rows).sort_values(
    ["Model order", "Covariate"],
    ignore_index=True,
)

max_abs_aic_difference = BASELINE_AUDIT["AIC audit difference"].abs().max()

print("\nAnalysis complete.")
print(f"Elapsed time: {(time.time() - analysis_start) / 60:.1f} minutes")
print(f"Models fitted: {len(BASELINE_AUDIT)}")
print(f"Covariate refits: {FULL_SCREEN['Estimable'].sum()}")
print(f"Maximum absolute baseline-AIC difference: {max_abs_aic_difference:.8f}")

if max_abs_aic_difference > 0.001:
    warnings.warn(
        "At least one refitted baseline AIC differs from the supplied AIC by "
        "more than 0.001. Check that the data and distribution conventions "
        "match the earlier analysis."
    )


[Model 01/41] PR_following_4W | Weibull | n=43
    [1/7] Target Vehicle Speed | ΔAIC= -1.992 | KS p=0.016 | AD p=0.036 | Fail: ΔAIC < 2 | 4.3s
    [2/7] Leading Vehicle Speed | ΔAIC= -1.998 | KS p=0.014 | AD p=0.044 | Fail: ΔAIC < 2 | 4.2s
    [3/7] Speed Difference | ΔAIC= -1.991 | KS p=0.014 | AD p=0.044 | Fail: ΔAIC < 2 | 4.3s
    [4/7] Occupancy | ΔAIC= -1.786 | KS p=0.052 | AD p=0.056 | Fail: ΔAIC < 2 | 4.5s
    [5/7] Off-centeredness | ΔAIC=  0.366 | KS p=0.154 | AD p=0.182 | Fail: ΔAIC < 2 | 4.6s
    [6/7] Site | ΔAIC= -1.838 | KS p=0.020 | AD p=0.016 | Fail: ΔAIC < 2 | 4.5s
    [7/7] Flow | ΔAIC= -1.402 | KS p=0.024 | AD p=0.056 | Fail: ΔAIC < 2 | 4.2s
[Model 02/41] PR_following_4W | Generalized gamma | n=43
    [1/7] Target Vehicle Speed | ΔAIC= -2.000 | KS p=0.324 | AD p=0.346 | Fail: ΔAIC < 2 | 20.5s
    [2/7] Leading Vehicle Speed | ΔAIC= -1.949 | KS p=0.304 | AD p=0.292 | Fail: ΔAIC < 2 | 20.8s
    [3/7] Speed Difference | ΔAIC= -1.938 | KS p=0.280 | AD p=0.270 | Fail: ΔAI


## Build the requested wide table and supporting p-value tables

In the main table, positive ΔAIC values indicate improvement. `Pass` applies
the stated rule: ΔAIC ≥ 2, and any KS/AD acceptance present at baseline must
remain accepted after covariate incorporation.


In [7]:

# ============================================================
# 7. Wide output matrices and covariate summary
# ============================================================

MODEL_INDEX = ["Model order", "Pair", "Distribution"]


def ordered_pivot(value_column, suffix):
    """Model × covariate wide table in the requested covariate order."""
    pivot = FULL_SCREEN.pivot(
        index=MODEL_INDEX,
        columns="Covariate label",
        values=value_column,
    )

    ordered_labels = [COVARIATE_LABELS[c] for c in COVARIATES]
    pivot = pivot.reindex(columns=ordered_labels)
    pivot.columns = [
        f"Covariate: {label} {suffix}"
        for label in ordered_labels
    ]
    return pivot.reset_index().sort_values("Model order", ignore_index=True)


baseline_for_wide = BASELINE_AUDIT[
    [
        "Model order",
        "Pair",
        "Distribution",
        "Reference AIC",
        "Refitted baseline AIC",
        "Baseline KS p",
        "Baseline AD p",
        "Baseline accepted by both",
    ]
].copy()


# Exact requested structure, with baseline AIC audit columns retained.
DELTA_AIC_WIDE = baseline_for_wide[
    [
        "Model order",
        "Pair",
        "Distribution",
        "Reference AIC",
        "Refitted baseline AIC",
    ]
].merge(
    ordered_pivot("ΔAIC", "ΔAIC"),
    on=MODEL_INDEX,
    how="left",
    validate="one_to_one",
)


KS_P_WIDE = baseline_for_wide[
    MODEL_INDEX + ["Baseline KS p"]
].merge(
    ordered_pivot("Covariate KS p", "KS p"),
    on=MODEL_INDEX,
    how="left",
    validate="one_to_one",
)


AD_P_WIDE = baseline_for_wide[
    MODEL_INDEX + ["Baseline AD p"]
].merge(
    ordered_pivot("Covariate AD p", "AD p"),
    on=MODEL_INDEX,
    how="left",
    validate="one_to_one",
)


KS_P_CHANGE_WIDE = baseline_for_wide[
    MODEL_INDEX + ["Baseline KS p"]
].merge(
    ordered_pivot("ΔKS p", "ΔKS p"),
    on=MODEL_INDEX,
    how="left",
    validate="one_to_one",
)


AD_P_CHANGE_WIDE = baseline_for_wide[
    MODEL_INDEX + ["Baseline AD p"]
].merge(
    ordered_pivot("ΔAD p", "ΔAD p"),
    on=MODEL_INDEX,
    how="left",
    validate="one_to_one",
)


LRT_P_WIDE = ordered_pivot("LR p", "LR p")
BETA_WIDE = ordered_pivot("β", "β")
EFFECT_RATIO_WIDE = ordered_pivot(
    "Acceleration ratio exp(β)",
    "exp(β)",
)


DECISION_WIDE = ordered_pivot("Decision", "Decision")


def summarize_covariate(group):
    estimable = group[group["Estimable"]]
    return pd.Series(
        {
            "Models listed": len(group),
            "Models estimable": len(estimable),
            "AIC improved (ΔAIC ≥ 2)": int(
                estimable["AIC improvement ≥ 2"].sum()
            ),
            "Passes stated rule": int(
                estimable["Passes stated rule"].sum()
            ),
            "Strict passes": int(
                estimable["Strict pass (AIC + both GOF)"].sum()
            ),
            "Median ΔAIC": estimable["ΔAIC"].median(),
            "Maximum ΔAIC": estimable["ΔAIC"].max(),
            "Median LR p": estimable["LR p"].median(),
            "Median ΔKS p": estimable["ΔKS p"].median(),
            "Median ΔAD p": estimable["ΔAD p"].median(),
        }
    )


COVARIATE_SUMMARY = (
    FULL_SCREEN.groupby(
        ["Covariate", "Covariate label"],
        sort=False,
        observed=True,
    )[
        [
            "Estimable",
            "AIC improvement ≥ 2",
            "Passes stated rule",
            "Strict pass (AIC + both GOF)",
            "ΔAIC",
            "LR p",
            "ΔKS p",
            "ΔAD p",
        ]
    ]
    .apply(summarize_covariate)
    .reset_index()
)

covariate_order_lookup = {
    covariate: position
    for position, covariate in enumerate(COVARIATES)
}
COVARIATE_SUMMARY["_order"] = COVARIATE_SUMMARY["Covariate"].map(
    covariate_order_lookup
)
COVARIATE_SUMMARY = (
    COVARIATE_SUMMARY
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


METHOD_NOTES = pd.DataFrame(
    {
        "Item": [
            "Models screened",
            "Covariate entry",
            "Distributional link",
            "Continuous coding",
            "Boolean coding",
            "Site coding",
            "AIC improvement",
            "Covariate significance",
            "Goodness-of-fit statistics",
            "Goodness-of-fit p-values",
            "Bootstrap refitting",
            "Bootstrap covariates",
            "Bootstrap replicates",
            "Acceptance threshold",
            "Stated selection rule",
            "Strict alternative",
            "AIC sign convention",
        ],
        "Specification": [
            f"{len(MODELS_TO_RUN)} supplied pair–distribution models",
            "Each covariate is entered separately; no multivariable model in this screen",
            "Log-scale accelerated failure-time model, T = exp(beta*z)Y",
            "Within-pair z-score; beta is per one pair-specific SD",
            "False=0 and True=1",
            "Shahjahanpur=0 and Tikatuli=1",
            f"Meaningful improvement when ΔAIC ≥ {DELTA_AIC_REQUIRED:.1f}",
            "Likelihood-ratio test of H0: beta=0 with 1 degree of freedom",
            "KS D and Anderson–Darling A² on conditional PIT values",
            "Parametric-bootstrap p-values with add-one correction",
            "The same baseline or covariate model is refitted in every replicate",
            "Observed covariate values are held fixed during simulation",
            str(N_BOOT),
            f"Accepted when p ≥ {ALPHA_GOF:.2f}",
            "ΔAIC ≥ 2 and no baseline-accepted KS/AD test becomes rejected",
            "ΔAIC ≥ 2 and the covariate model is accepted by both KS and AD",
            "ΔAIC = baseline AIC − covariate-model AIC; positive values favour the covariate",
        ],
    }
)


print("Requested ΔAIC table")
display(
    DELTA_AIC_WIDE.drop(columns="Model order").round(3)
)

print("Covariate summary")
display(COVARIATE_SUMMARY.round(4))


NameError: name 'BASELINE_AUDIT' is not defined

In [8]:

# ============================================================
# 8. Export all results to one formatted Excel workbook
# ============================================================

from openpyxl import load_workbook
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


export_tables = {
    "S0_Method": METHOD_NOTES,
    "S1_Delta_AIC": DELTA_AIC_WIDE.drop(columns="Model order"),
    "S2_KS_p": KS_P_WIDE.drop(columns="Model order"),
    "S3_AD_p": AD_P_WIDE.drop(columns="Model order"),
    "S4_Delta_KS_p": KS_P_CHANGE_WIDE.drop(columns="Model order"),
    "S5_Delta_AD_p": AD_P_CHANGE_WIDE.drop(columns="Model order"),
    "S6_LRT_p": LRT_P_WIDE.drop(columns="Model order"),
    "S7_Beta": BETA_WIDE.drop(columns="Model order"),
    "S8_Effect_ratio": EFFECT_RATIO_WIDE.drop(columns="Model order"),
    "S9_Decisions": DECISION_WIDE.drop(columns="Model order"),
    "S10_Full_screen": FULL_SCREEN.drop(columns="Model order"),
    "S11_Baseline_audit": BASELINE_AUDIT.drop(columns="Model order"),
    "S12_Cov_summary": COVARIATE_SUMMARY,
}

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, table in export_tables.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)


# ---------- compact workbook formatting ----------
workbook = load_workbook(OUTPUT_XLSX)
header_fill = PatternFill("solid", fgColor="1B3B6F")
header_font = Font(color="FFFFFF", bold=True)
green_fill = PatternFill("solid", fgColor="E2F0D9")
red_fill = PatternFill("solid", fgColor="FCE4D6")

for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    worksheet.sheet_view.showGridLines = False
    worksheet.row_dimensions[1].height = 34

    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for column_cells in worksheet.columns:
        column_index = column_cells[0].column
        header = str(column_cells[0].value or "")
        max_length = max(
            (
                len(str(cell.value))
                for cell in column_cells
                if cell.value is not None
            ),
            default=8,
        )
        width_cap = 42 if "Coding" in header or "message" in header else 25
        worksheet.column_dimensions[
            get_column_letter(column_index)
        ].width = min(max(max_length + 2, 11), width_cap)

        if any(
            token in header
            for token in [
                "AIC", "logLik", "KS", "AD", "LR", "β",
                "exp(β)", "Median", "Maximum",
            ]
        ):
            for cell in column_cells[1:]:
                cell.number_format = "0.0000"

    # ΔAIC cells: green at ≥2, pale red below 2.
    if worksheet.title == "S1_Delta_AIC":
        for cell in worksheet[1]:
            if cell.value and str(cell.value).endswith("ΔAIC"):
                column_letter = get_column_letter(cell.column)
                data_range = (
                    f"{column_letter}2:{column_letter}{worksheet.max_row}"
                )
                worksheet.conditional_formatting.add(
                    data_range,
                    CellIsRule(
                        operator="greaterThanOrEqual",
                        formula=[str(DELTA_AIC_REQUIRED)],
                        fill=green_fill,
                    ),
                )
                worksheet.conditional_formatting.add(
                    data_range,
                    CellIsRule(
                        operator="lessThan",
                        formula=[str(DELTA_AIC_REQUIRED)],
                        fill=red_fill,
                    ),
                )

    if worksheet.title == "S9_Decisions":
        for row in worksheet.iter_rows(min_row=2):
            for cell in row:
                if cell.value == "Pass":
                    cell.fill = green_fill
                elif isinstance(cell.value, str) and cell.value.startswith("Fail"):
                    cell.fill = red_fill

workbook.save(OUTPUT_XLSX)

print(f"Wrote: {OUTPUT_XLSX}")


Wrote: D:\Headway\Tables\T4_41_Distribution_Covariate_Screening.xlsx



## Reading the outputs

- `S1_Delta_AIC` is the requested wide table. A value ≥ 2 indicates meaningful
  AIC improvement.
- `S2_KS_p` and `S3_AD_p` compare baseline and covariate-model bootstrap
  p-values.
- `S4_Delta_KS_p` and `S5_Delta_AD_p` show the numerical changes in p-values.
- `S6_LRT_p` tests the covariate coefficient.
- `S7_Beta` and `S8_Effect_ratio` show direction and effect size.
- `S9_Decisions` applies the stated AIC-plus-GOF-preservation rule.
- `S10_Full_screen` contains every statistic and diagnostic for all
  \(41\times7=287\) attempted combinations.
- `S11_Baseline_audit` verifies the supplied AIC values.

A positive \(\beta\) means longer headway; a negative \(\beta\) means shorter
headway. For a continuous variable, `exp(β)` is the headway multiplier for a
one-within-pair-SD increase. For a binary variable, it is the multiplier for
1 relative to 0.


In [8]:
# ============================================================
# MAXIMUM SIGNIFICANT ΔAIC BY PAIR AND COVARIATE
# Rows: target–leader pairs
# Columns: covariates
# Values: maximum qualifying ΔAIC across distributions
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


# ------------------------------------------------------------
# 1. Selection criteria
# ------------------------------------------------------------

ALPHA_LR = 0.05
MIN_DELTA_AIC = 2.0

# True applies your earlier requirement:
# a baseline-accepted KS/AD result must not become rejected.
REQUIRE_GOF_PRESERVED = True


COVARIATE_ORDER = [
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Occupancy",
    "Off_centeredness",
    "Site",
    "Flow_pcu/hr/m",
]

COVARIATE_DISPLAY = {
    "Target_Speed_km/hr": "Target Vehicle Speed",
    "Leading_Speed_km/hr": "Leading Vehicle Speed",
    "Speed_Difference": "Speed Difference",
    "Occupancy": "Occupancy",
    "Off_centeredness": "Off-centeredness",
    "Site": "Site",
    "Flow_pcu/hr/m": "Flow",
}


# ------------------------------------------------------------
# 2. Load the full covariate-screening table
# ------------------------------------------------------------

# If this cell is run directly after the screening notebook,
# FULL_SCREEN will already be available.
if (
    "FULL_SCREEN" in globals()
    and isinstance(FULL_SCREEN, pd.DataFrame)
):
    screening_table = FULL_SCREEN.copy()

else:
    # Otherwise, read the exported screening workbook.
    possible_files = []

    if "OUTPUT_XLSX" in globals():
        possible_files.append(Path(OUTPUT_XLSX))

    possible_files.extend(
        [
            Path("Tables/T4_41_Distribution_Covariate_Screening.xlsx"),
            Path("T4_41_Distribution_Covariate_Screening.xlsx"),
        ]
    )

    input_file = next(
        (
            file
            for file in possible_files
            if file.exists()
        ),
        None,
    )

    if input_file is None:
        raise FileNotFoundError(
            "The covariate-screening workbook was not found. "
            "Set input_file to the correct path."
        )

    screening_table = pd.read_excel(
        input_file,
        sheet_name="S10_Full_screen",
    )


# ------------------------------------------------------------
# 3. Validate and clean the required columns
# ------------------------------------------------------------

required_columns = [
    "Pair",
    "Distribution",
    "Covariate",
    "Estimable",
    "ΔAIC",
    "β",
    "LR p",
    "GOF acceptance preserved",
    "Optimizer converged",
]

missing_columns = [
    column
    for column in required_columns
    if column not in screening_table.columns
]

if missing_columns:
    raise KeyError(
        "The following required columns are missing: "
        + ", ".join(missing_columns)
    )


# Ensure numeric columns are numeric.
for column in ["ΔAIC", "β", "LR p"]:
    screening_table[column] = pd.to_numeric(
        screening_table[column],
        errors="coerce",
    )


def to_boolean(series):
    """
    Convert Boolean, 0/1, or text Boolean values into True/False.
    """
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


estimable = to_boolean(
    screening_table["Estimable"]
)

converged = to_boolean(
    screening_table["Optimizer converged"]
)

gof_preserved = to_boolean(
    screening_table["GOF acceptance preserved"]
)


# ------------------------------------------------------------
# 4. Select statistically meaningful improvements
# ------------------------------------------------------------

selection_mask = (
    estimable
    & converged
    & screening_table["ΔAIC"].ge(MIN_DELTA_AIC)
    & screening_table["LR p"].lt(ALPHA_LR)
    & screening_table["β"].notna()
)

if REQUIRE_GOF_PRESERVED:
    selection_mask &= gof_preserved


qualifying_results = screening_table.loc[
    selection_mask
].copy()


print(
    f"Qualifying distribution–covariate models: "
    f"{len(qualifying_results)}"
)


# ------------------------------------------------------------
# 5. Find maximum ΔAIC for every pair × covariate
# ------------------------------------------------------------

pair_order = (
    screening_table["Pair"]
    .drop_duplicates()
    .tolist()
)

if qualifying_results.empty:

    winner_details = pd.DataFrame(
        columns=[
            "Pair",
            "Covariate",
            "Distribution",
            "ΔAIC",
            "LR p",
            "β",
        ]
    )

else:

    winner_indices = (
        qualifying_results
        .groupby(
            ["Pair", "Covariate"],
            sort=False,
            observed=True,
        )["ΔAIC"]
        .idxmax()
    )

    winner_details = (
        qualifying_results.loc[
            winner_indices
        ]
        .copy()
        .sort_values(
            ["Pair", "Covariate"],
            ignore_index=True,
        )
    )


# ------------------------------------------------------------
# 6. Requested maximum-ΔAIC matrix
# ------------------------------------------------------------

maximum_delta_aic_matrix = (
    winner_details
    .pivot(
        index="Pair",
        columns="Covariate",
        values="ΔAIC",
    )
    .reindex(
        index=pair_order,
        columns=COVARIATE_ORDER,
    )
    .rename(
        columns=COVARIATE_DISPLAY
    )
)

maximum_delta_aic_matrix.index.name = "Pair"


# ------------------------------------------------------------
# 7. Companion matrix: distribution producing maximum ΔAIC
# ------------------------------------------------------------

winning_distribution_matrix = (
    winner_details
    .pivot(
        index="Pair",
        columns="Covariate",
        values="Distribution",
    )
    .reindex(
        index=pair_order,
        columns=COVARIATE_ORDER,
    )
    .rename(
        columns=COVARIATE_DISPLAY
    )
)

winning_distribution_matrix.index.name = "Pair"


# ------------------------------------------------------------
# 8. Detailed winning-model table
# ------------------------------------------------------------

winner_details_table = winner_details[
    [
        "Pair",
        "Covariate",
        "Distribution",
        "ΔAIC",
        "LR p",
        "β",
        "Acceleration ratio exp(β)",
        "Covariate KS p",
        "Covariate AD p",
        "GOF acceptance preserved",
    ]
].copy()

winner_details_table["Covariate"] = (
    winner_details_table["Covariate"]
    .map(COVARIATE_DISPLAY)
)

winner_details_table = (
    winner_details_table
    .rename(
        columns={
            "Covariate": "Covariate producing maximum ΔAIC",
            "Distribution": "Winning distribution",
        }
    )
    .sort_values(
        [
            "Pair",
            "Covariate producing maximum ΔAIC",
        ],
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# 9. Identify the strongest covariate for each pair
# ------------------------------------------------------------

pair_best_rows = []

for pair in maximum_delta_aic_matrix.index:

    pair_values = maximum_delta_aic_matrix.loc[pair].dropna()

    if pair_values.empty:
        pair_best_rows.append(
            {
                "Pair": pair,
                "Strongest covariate": "None",
                "Maximum ΔAIC": np.nan,
                "Winning distribution": "None",
            }
        )
        continue

    strongest_covariate = pair_values.idxmax()
    maximum_delta = pair_values.max()

    winning_distribution = (
        winning_distribution_matrix.loc[
            pair,
            strongest_covariate,
        ]
    )

    pair_best_rows.append(
        {
            "Pair": pair,
            "Strongest covariate": strongest_covariate,
            "Maximum ΔAIC": maximum_delta,
            "Winning distribution": winning_distribution,
        }
    )

pair_best_covariate = pd.DataFrame(
    pair_best_rows
)


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print(
    "\nMaximum significant ΔAIC by pair and covariate\n"
    f"Criteria: ΔAIC ≥ {MIN_DELTA_AIC}, "
    f"LR p < {ALPHA_LR}"
    + (
        ", and GOF acceptance preserved"
        if REQUIRE_GOF_PRESERVED
        else ""
    )
)

display(
    maximum_delta_aic_matrix.style
    .format(
        precision=3,
        na_rep="—",
    )
    .highlight_max(
        axis=1,
        color="#C6E0B4",
    )
)


print("\nDistribution producing each maximum ΔAIC")
display(
    winning_distribution_matrix.fillna("—")
)


print("\nStrongest covariate for each pair")
display(
    pair_best_covariate.round(
        {
            "Maximum ΔAIC": 3,
        }
    )
)


print("\nDetailed winning models")
display(
    winner_details_table.round(
        {
            "ΔAIC": 3,
            "LR p": 4,
            "β": 4,
            "Acceleration ratio exp(β)": 4,
            "Covariate KS p": 4,
            "Covariate AD p": 4,
        }
    )
)


# ------------------------------------------------------------
# 11. Export results
# ------------------------------------------------------------

if "OUTPUT_DIR" in globals():
    summary_output = (
        Path(OUTPUT_DIR)
        / "T4_Max_Covariate_Effect_by_Pair.xlsx"
    )
else:
    summary_output = Path(
        "T4_Max_Covariate_Effect_by_Pair.xlsx"
    )

summary_output.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with pd.ExcelWriter(
    summary_output,
    engine="openpyxl",
) as writer:

    maximum_delta_aic_matrix.reset_index().to_excel(
        writer,
        sheet_name="Maximum_dAIC",
        index=False,
    )

    winning_distribution_matrix.reset_index().to_excel(
        writer,
        sheet_name="Winning_distribution",
        index=False,
    )

    pair_best_covariate.to_excel(
        writer,
        sheet_name="Pair_best_covariate",
        index=False,
    )

    winner_details_table.to_excel(
        writer,
        sheet_name="Winner_details",
        index=False,
    )


print(f"\nWrote: {summary_output}")

FileNotFoundError: The covariate-screening workbook was not found. Set input_file to the correct path.